**Instalación de dependencias**

Agrega los paquetes necesarios a tu proyecto:

In [ ]:
%pip install azure-ai-inference[opentelemetry]
%pip install azure-search-documents 
%pip install azure-identity
%pip install openai

**Configuración de la aplicación**

Aquí está la configuración de tu aplicación, podrías mejorarla usando un archivo .env en lugar de dejar aquí todas tus variables!
¿Puedes con eso?

In [ ]:
import os
from openai import AzureOpenAI
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
import dotenv
from azure.search.documents.models import VectorizedQuery, VectorizableTextQuery

dotenv.load_dotenv()

OPENAI_API_TYPE="azure"
AZURE_OPENAI_API_KEY="<coloca aquí tu clave de Azure OpenAI>"
AZURE_OPENAI_ENDPOINT="https://openai-rag-platzi.openai.azure.com/"
OPENAI_API_VERSION="2025-01-01-preview"

AZURE_OPENAI_EMBEDDING_DEPLOYED_MODEL_NAME="text-embedding-ada-002"

AZURE_SEARCH_SERVICE_ENDPOINT="https://testingitplatzi.search.windows.net"
AZURE_SEARCH_INDEX_NAME="rag-testing2"
AZURE_SEARCH_ADMIN_KEY="<coloca aquí tu clave de Azure Search>"

openai_client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=OPENAI_API_VERSION
)

search_client = SearchClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX_NAME,
    credential=AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
)

def get_embedding(text):
    return openai_client.embeddings.create(
        model=AZURE_OPENAI_EMBEDDING_DEPLOYED_MODEL_NAME,
        input=text
    ).data[0].embedding

**Haz la pregunta**

Pregúntale algo a tu RAG

In [ ]:
user_question = "What is included in my Northwind Health Plus plan that is not in standard?"
user_question_vector = get_embedding(user_question)
print(user_question_vector)

**Muestra la pregunta en un modo que se puede leer**

Convierte los resultados de tu RAG en un formato que se pueda leer.

In [ ]:
search_results = search_client.search(
    None,
    top=3,
    vector_queries=[
        VectorizableTextQuery( 
            text=user_question, k_nearest_neighbors=3, fields="text_vector"
        )
    ],
)

for result in search_results:
    print("Chunk ID:", result["chunk_id"])
    print("Title:", result["title"])
    print("Text:", result["chunk"])
    print()

**Agrega un LLM a tu RAG**

Muestra cómo puedes agregar un modelo de lenguaje a tu RAG para mejorar la calidad de las respuestas.

In [ ]:
AZURE_OPENAI_CHAT_COMPLETION_DEPLOYED_MODEL_NAME="gpt-4o"

context = ""
for result in search_results:
    context += result["chunk"] + "\n\n"

SYSTEM_MESSAGE = f"""
You are an AI Assistant.
Be brief in your answers. Answer ONLY with the facts listed in the retrieved text.

Context:
{context}
"""

USER_MESSAGE = user_question

response = openai_client.chat.completions.create(
    model=AZURE_OPENAI_CHAT_COMPLETION_DEPLOYED_MODEL_NAME,
    temperature=0.7,
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

answer = response.choices[0].message.content
print(answer)